# Production Inference: vLLM, TensorRT-LLM & Quantization

## The Cost Crisis

Running Llama 2 70B on AWS:
- **p4d.24xlarge** (8x A100): $32.77/hour = **$23,800/month**
- **With vLLM + quantization**: $2,000/month (12x cheaper)

This module teaches you to build high-throughput, low-cost inference systems.

## Part 1: vLLM - Paged Attention Revolution

### The Problem: KV Cache Explosion

In standard transformer inference, the **key-value cache** grows linearly with sequence length:

$$\text{KV Cache Size} = 2 \times \text{batch\_size} \times \text{seq\_len} \times \text{hidden\_dim}$$

For a 70B model with batch_size=32, seq_len=2048:
$$\text{KV Cache} = 2 \times 32 \times 2048 \times 8192 \approx 1 \text{ GB per token}$$

This wastes GPU memory and causes **fragmentation**.

### The Solution: Paged Attention

vLLM implements **Paged Attention**, treating KV cache like virtual memory:



**Performance Gains:**
- **Throughput:** 10-20x higher than standard inference
- **Latency:** 50-100ms per token (vs 200-500ms)
- **Memory:** 2-4x more efficient

### Benchmark: vLLM vs Standard



---

In [ ]:
from vllm import LLM, SamplingParams

# Initialize vLLM with paged attention
llm = LLM(
    model="meta-llama/Llama-2-70b-hf",
    tensor_parallel_size=8,  # Distribute across 8 GPUs
    gpu_memory_utilization=0.9,  # Use 90% of GPU memory
    enable_prefix_caching=True,  # Cache repeated prefixes
)

# Batch inference (vLLM handles scheduling)
prompts = [
    "What is machine learning?",
    "Explain deep learning",
    "Define neural networks",
] * 100  # 300 prompts

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.95,
    max_tokens=256,
)

# vLLM automatically batches and schedules
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    print(output.outputs[0].text)

In [ ]:
import time
import torch

# Standard inference (naive batching)
def standard_inference(model, prompts, batch_size=1):
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        with torch.no_grad():
            output = model.generate(batch, max_length=256)
        outputs.extend(output)
    return outputs

# vLLM inference
def vllm_inference(llm, prompts):
    return llm.generate(prompts, SamplingParams(max_tokens=256))

# Benchmark
prompts = ["What is AI?"] * 1000

start = time.time()
standard_inference(model, prompts, batch_size=32)
standard_time = time.time() - start

start = time.time()
vllm_inference(llm, prompts)
vllm_time = time.time() - start

print(f"Standard: {standard_time:.2f}s")
print(f"vLLM: {vllm_time:.2f}s")
print(f"Speedup: {standard_time / vllm_time:.1f}x")

## Part 2: TensorRT-LLM - NVIDIA's Optimized Runtime

### Compilation to CUDA Kernels

TensorRT-LLM compiles models to optimized CUDA kernels:

```bash
# Install TensorRT-LLM
pip install tensorrt-llm

# Convert Hugging Face model to TensorRT format
python -m tensorrt_llm.commands.convert \
    --model_dir ./llama-2-70b \
    --output_dir ./llama-2-70b-trt \
    --tp_size 8 \
    --pp_size 1
```

### Inference with TensorRT-LLM



**Performance vs vLLM:**
- **Latency:** 20-30% faster than vLLM
- **Throughput:** Similar to vLLM
- **Setup:** More complex (requires compilation)

---

In [ ]:
from tensorrt_llm.runtime import ModelRunner

# Load compiled model
runner = ModelRunner.from_dir(
    engine_dir="./llama-2-70b-trt",
    rank=0,  # GPU rank
    debug_mode=False,
)

# Prepare input
input_ids = tokenizer.encode("What is AI?")
input_ids = torch.tensor([input_ids]).cuda()

# Generate
output_ids = runner.generate(
    input_ids,
    max_new_tokens=256,
    temperature=0.7,
)

output_text = tokenizer.decode(output_ids[0])
print(output_text)

## Part 3: Quantization - The Cost Killer

### INT8 Quantization (8-bit)

Reduce model size by 4x with minimal accuracy loss:



### GGUF Format (Quantized Weights)

For local inference, use GGUF (Ollama format):

```bash
# Convert to GGUF with Q4 quantization
python convert.py \
    --model-dir ./llama-2-70b \
    --output-type q4_k_m \
    --outfile ./llama-2-70b.gguf

# Run locally with Ollama
ollama create my-model -f Modelfile
ollama run my-model "What is AI?"
```

**Quantization Levels:**

| Format | Size | Speed | Accuracy |
|--------|------|-------|----------|
| **FP32** | 280 GB | Baseline | 100% |
| **FP16** | 140 GB | 1.5x | 99.9% |
| **INT8** | 70 GB | 2x | 99.5% |
| **INT4** | 35 GB | 3x | 98% |
| **GGUF Q4** | 35 GB | 4x | 97% |

### Cost Impact



---

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# 8-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-70b-hf",
    quantization_config=quantization_config,
    device_map="auto",
)

# Model is now 70B → 17.5B (4x smaller)
print(f"Model size: {model.get_memory_footprint() / 1e9:.1f} GB")

In [ ]:
# Llama 2 70B inference costs

# Standard (FP16)
cost_standard = 280 * 0.0001  # $0.028 per GB-hour
print(f"Standard: ${cost_standard:.3f}/hour")

# INT8 quantized
cost_int8 = 70 * 0.0001
print(f"INT8: ${cost_int8:.3f}/hour")

# GGUF on local GPU
cost_local = 0.01  # Electricity only
print(f"Local GGUF: ${cost_local:.3f}/hour")

print(f"Savings: {cost_standard / cost_int8:.1f}x cheaper with INT8")

## Part 4: Production Architecture

### Deployment: vLLM + Kubernetes

```yaml
# deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm-inference
spec:
  replicas: 3
  selector:
    matchLabels:
      app: vllm
  template:
    metadata:
      labels:
        app: vllm
    spec:
      containers:
      - name: vllm
        image: vllm/vllm-openai:latest
        args:
          - --model
          - meta-llama/Llama-2-70b-hf
          - --tensor-parallel-size
          - "8"
          - --gpu-memory-utilization
          - "0.9"
        resources:
          limits:
            nvidia.com/gpu: "8"
        ports:
        - containerPort: 8000
---
apiVersion: v1
kind: Service
metadata:
  name: vllm-service
spec:
  selector:
    app: vllm
  ports:
  - port: 8000
    targetPort: 8000
  type: LoadBalancer
```

### Load Testing



---

In [ ]:
import asyncio
import aiohttp
import time

async def load_test(num_requests=1000, concurrency=50):
    """Simulate production load"""
    async with aiohttp.ClientSession() as session:
        tasks = []
        start = time.time()
        
        for i in range(num_requests):
            task = session.post(
                "http://localhost:8000/v1/completions",
                json={
                    "model": "llama-2-70b",
                    "prompt": "What is AI?",
                    "max_tokens": 256,
                },
                timeout=aiohttp.ClientTimeout(total=60),
            )
            tasks.append(task)
            
            # Limit concurrency
            if len(tasks) >= concurrency:
                await asyncio.gather(*tasks)
                tasks = []
        
        if tasks:
            await asyncio.gather(*tasks)
        
        elapsed = time.time() - start
        throughput = num_requests / elapsed
        print(f"Throughput: {throughput:.1f} req/s")
        print(f"Total time: {elapsed:.1f}s")

# Run
asyncio.run(load_test(num_requests=1000, concurrency=50))

## Comparison: Which to Use?

| Tool | Throughput | Latency | Setup | Cost |
|------|-----------|---------|-------|------|
| **vLLM** | 10-20x | 50-100ms | Easy | Medium |
| **TensorRT-LLM** | 15-25x | 20-50ms | Hard | Medium |
| **Quantized (INT8)** | 5-10x | 100-200ms | Easy | Low |
| **GGUF Local** | 2-5x | 200-500ms | Easy | Very Low |

**Recommendation:**
- **Prototyping:** vLLM
- **Production (cloud):** vLLM + INT8
- **Production (on-prem):** TensorRT-LLM + INT8
- **Cost-sensitive:** GGUF quantization

---

## Key Concepts

| Concept | Impact | Complexity |
|---------|--------|-----------|
| **Paged Attention** | 10-20x throughput | Medium |
| **Tensor Parallelism** | Scale to 8+ GPUs | Medium |
| **INT8 Quantization** | 4x smaller, 2x faster | Low |
| **Prefix Caching** | 2-3x faster for repeated prompts | Low |
| **Batch Scheduling** | Maximize GPU utilization | Medium |

---

## Quizzes

### Quiz 1: KV Cache Problem
**Question:** Why does KV cache grow with sequence length?
- A) Attention mechanism stores key-value pairs for each token ✓
- B) Model weights increase
- C) Batch size increases
- D) GPU memory is limited

### Quiz 2: vLLM Speedup
**Question:** vLLM achieves 10-20x speedup primarily through:
- A) Paged Attention + efficient batching ✓
- B) Better algorithms
- C) Faster GPUs
- D) Smaller models

### Quiz 3: Quantization Tradeoff
**Question:** INT8 quantization reduces model size 4x. What's the main tradeoff?
- A) Slight accuracy loss (~0.5%) ✓
- B) Slower inference
- C) Requires retraining
- D) Only works for small models

---

## Resources & References

- **[vLLM Documentation](https://docs.vllm.ai/)** - Paged Attention inference
- **[TensorRT-LLM](https://github.com/NVIDIA/TensorRT-LLM)** - NVIDIA's optimized runtime
- **[BitsAndBytes](https://github.com/TimDettmers/bitsandbytes)** - Quantization library
- **[GGUF Format](https://github.com/ggerganov/ggml)** - Quantized weights
- **[Papers with Code: Inference Optimization](https://paperswithcode.com/)** - Latest research